# Teklovossen Gameplay Simulator

This notebook simulates multiple turns of Teklovossen gameplay to collect data for card balance analysis and ML training.

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Any
import json
from datetime import datetime
import random
from collections import defaultdict

# Import our simulation engine
from simulation.game_engine import GameEngine, GameState, Player, Card
from simulation.card_database import create_card_database, create_deck_preset, create_opponent_deck, CARD_BALANCE_SCORES

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print("Teklovossen Gameplay Simulator initialized!")

## Setup Card Database and Decks

In [ ]:
# Initialize card database
card_db = create_card_database()
print(f"Card database loaded with {len(card_db)} cards")

# Display card themes
themes = defaultdict(int)
for card in card_db.values():
    themes[card.theme] += 1

print("\nCard distribution by theme:")
for theme, count in themes.items():
    print(f"  {theme}: {count} cards")

# Create deck presets
deck_presets = {
    'nano_focused': create_deck_preset('nano_focused', card_db),
    'ai_focused': create_deck_preset('ai_focused', card_db),
    'quantum_focused': create_deck_preset('quantum_focused', card_db),
    'balanced_mix': create_deck_preset('balanced_mix', card_db)
}

opponent_deck = create_opponent_deck(card_db)

print(f"\nDeck presets created:")
for name, deck in deck_presets.items():
    print(f"  {name}: {len(deck)} cards")

## Single Game Simulation

In [ ]:
def simulate_single_game(deck_name: str, max_turns: int = 20) -> Dict[str, Any]:
    """Simulate a single game and return detailed statistics"""
    
    engine = GameEngine()
    teklo_deck = deck_presets[deck_name].copy()
    opponent = opponent_deck.copy()
    
    # Initialize game
    game_state = engine.initialize_game(teklo_deck, opponent)
    
    # Track game statistics
    game_stats = {
        'deck_name': deck_name,
        'turns_played': 0,
        'winner': None,
        'final_life': {'teklovossen': 20, 'opponent': 20},
        'cards_played': {'teklovossen': [], 'opponent': []},
        'resources_generated': {'teklo_energy': 0, 'nanite_counters': 0, 'quantum_charges': 0},
        'equipment_equipped': [],
        'damage_dealt': {'teklovossen': 0, 'opponent': 0},
        'evos_equipped': 0,
        'theme_synergies': defaultdict(int),
        'turn_data': []
    }
    
    # Simulate turns
    while game_stats['turns_played'] < max_turns and not engine.is_game_over():
        turn_data = engine.simulate_turn()
        game_stats['turns_played'] += 1
        game_stats['turn_data'].append(turn_data)
        
        # Track statistics for current player
        current_player = game_state.current_player
        if current_player.name == "Teklovossen":
            # Track equipment and evos
            for equipment in current_player.equipment.values():
                if equipment:
                    if equipment.is_evo:
                        game_stats['evos_equipped'] += 1
                    game_stats['theme_synergies'][equipment.theme] += 1
            
            # Track resources
            for resource, amount in current_player.resources.items():
                if resource.value in game_stats['resources_generated']:
                    game_stats['resources_generated'][resource.value] += amount
    
    # Final game state
    game_stats['final_life']['teklovossen'] = game_state.players[0].life
    game_stats['final_life']['opponent'] = game_state.players[1].life
    
    if engine.is_game_over():
        winner = engine.get_winner()
        game_stats['winner'] = winner.name if winner else 'draw'
    else:
        game_stats['winner'] = 'timeout'
    
    return game_stats

# Run a single test game
test_game = simulate_single_game('nano_focused', max_turns=10)
print(f"Test game completed:")
print(f"  Deck: {test_game['deck_name']}")
print(f"  Turns: {test_game['turns_played']}")
print(f"  Winner: {test_game['winner']}")
print(f"  Final Life: {test_game['final_life']}")
print(f"  Evos Equipped: {test_game['evos_equipped']}")
print(f"  Theme Synergies: {dict(test_game['theme_synergies'])}")

## Batch Simulation and Data Collection

In [ ]:
def run_batch_simulation(games_per_deck: int = 50) -> pd.DataFrame:
    """Run multiple simulations across different deck configurations"""
    
    all_results = []
    
    print(f"Running batch simulation: {games_per_deck} games per deck type...")
    
    for deck_name in deck_presets.keys():
        print(f"  Simulating {deck_name}...")
        
        for game_num in range(games_per_deck):
            if game_num % 10 == 0:
                print(f"    Game {game_num + 1}/{games_per_deck}")
            
            game_result = simulate_single_game(deck_name, max_turns=15)
            
            # Flatten results for DataFrame
            flattened_result = {
                'game_id': f"{deck_name}_{game_num}",
                'deck_name': game_result['deck_name'],
                'turns_played': game_result['turns_played'],
                'winner': game_result['winner'],
                'teklovossen_final_life': game_result['final_life']['teklovossen'],
                'opponent_final_life': game_result['final_life']['opponent'],
                'evos_equipped': game_result['evos_equipped'],
                'teklo_energy_generated': game_result['resources_generated']['teklo_energy'],
                'nanite_counters_generated': game_result['resources_generated']['nanite_counters'],
                'quantum_charges_generated': game_result['resources_generated']['quantum_charges'],
                'ai_theme_synergy': game_result['theme_synergies']['AI'],
                'nano_theme_synergy': game_result['theme_synergies']['Nano'],
                'quantum_theme_synergy': game_result['theme_synergies']['Quantum'],
                'win_rate': 1 if game_result['winner'] == 'Teklovossen' else 0
            }
            
            all_results.append(flattened_result)
    
    results_df = pd.DataFrame(all_results)
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_df.to_csv(f"../data/simulations/batch_simulation_{timestamp}.csv", index=False)
    
    print(f"Batch simulation completed! Results saved to data/simulations/")
    return results_df

# Run batch simulation (start with smaller number for testing)
simulation_results = run_batch_simulation(games_per_deck=25)
print(f"\nSimulation completed with {len(simulation_results)} games total")

## Data Analysis and Visualization

In [ ]:
# Performance analysis by deck type
deck_performance = simulation_results.groupby('deck_name').agg({
    'win_rate': 'mean',
    'turns_played': 'mean',
    'evos_equipped': 'mean',
    'teklo_energy_generated': 'mean',
    'nanite_counters_generated': 'mean',
    'quantum_charges_generated': 'mean',
    'teklovossen_final_life': 'mean'
}).round(2)

print("Deck Performance Summary:")
print(deck_performance)

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Win Rate by Deck
sns.barplot(data=simulation_results, x='deck_name', y='win_rate', ax=axes[0,0])
axes[0,0].set_title('Win Rate by Deck Type')
axes[0,0].set_ylabel('Win Rate')
axes[0,0].tick_params(axis='x', rotation=45)

# Average Game Length
sns.boxplot(data=simulation_results, x='deck_name', y='turns_played', ax=axes[0,1])
axes[0,1].set_title('Game Length Distribution')
axes[0,1].set_ylabel('Turns Played')
axes[0,1].tick_params(axis='x', rotation=45)

# Resource Generation Efficiency
resource_cols = ['teklo_energy_generated', 'nanite_counters_generated', 'quantum_charges_generated']
resource_data = simulation_results.groupby('deck_name')[resource_cols].mean()
resource_data.plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Resource Generation by Deck')
axes[1,0].set_ylabel('Average Resources Generated')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Evo Equipment Usage
sns.violinplot(data=simulation_results, x='deck_name', y='evos_equipped', ax=axes[1,1])
axes[1,1].set_title('Evo Equipment Usage')
axes[1,1].set_ylabel('Evos Equipped')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Statistical summary
print("\nStatistical Summary:")
print(simulation_results.describe())

## Card Performance Analysis

In [ ]:
def analyze_card_performance(results_df: pd.DataFrame) -> pd.DataFrame:
    """Analyze individual card performance from simulation results"""
    
    # Extract card usage patterns by theme
    card_analysis = []
    
    for deck_name in results_df['deck_name'].unique():
        deck_data = results_df[results_df['deck_name'] == deck_name]
        
        analysis = {
            'deck_type': deck_name,
            'sample_size': len(deck_data),
            'avg_win_rate': deck_data['win_rate'].mean(),
            'avg_game_length': deck_data['turns_played'].mean(),
            'avg_evos_equipped': deck_data['evos_equipped'].mean(),
            'resource_efficiency': (
                deck_data['teklo_energy_generated'].mean() + 
                deck_data['nanite_counters_generated'].mean() + 
                deck_data['quantum_charges_generated'].mean()
            ) / deck_data['turns_played'].mean(),
            'survivability': deck_data['teklovossen_final_life'].mean(),
            'theme_focus': {
                'AI': deck_data['ai_theme_synergy'].mean(),
                'Nano': deck_data['nano_theme_synergy'].mean(),
                'Quantum': deck_data['quantum_theme_synergy'].mean()
            }
        }
        
        card_analysis.append(analysis)
    
    return pd.DataFrame(card_analysis)

# Analyze card performance
card_performance = analyze_card_performance(simulation_results)
print("Card Performance Analysis:")
print(card_performance.round(3))

# Create performance metrics visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Win Rate vs Resource Efficiency
axes[0].scatter(card_performance['resource_efficiency'], card_performance['avg_win_rate'], s=100)
for i, deck in enumerate(card_performance['deck_type']):
    axes[0].annotate(deck, (card_performance['resource_efficiency'].iloc[i], 
                           card_performance['avg_win_rate'].iloc[i]))
axes[0].set_xlabel('Resource Efficiency (Resources/Turn)')
axes[0].set_ylabel('Average Win Rate')
axes[0].set_title('Win Rate vs Resource Efficiency')

# Game Length vs Survivability
axes[1].scatter(card_performance['avg_game_length'], card_performance['survivability'], s=100)
for i, deck in enumerate(card_performance['deck_type']):
    axes[1].annotate(deck, (card_performance['avg_game_length'].iloc[i], 
                           card_performance['survivability'].iloc[i]))
axes[1].set_xlabel('Average Game Length (Turns)')
axes[1].set_ylabel('Average Final Life')
axes[1].set_title('Game Length vs Survivability')

# Evo Usage vs Win Rate
axes[2].scatter(card_performance['avg_evos_equipped'], card_performance['avg_win_rate'], s=100)
for i, deck in enumerate(card_performance['deck_type']):
    axes[2].annotate(deck, (card_performance['avg_evos_equipped'].iloc[i], 
                           card_performance['avg_win_rate'].iloc[i]))
axes[2].set_xlabel('Average Evos Equipped')
axes[2].set_ylabel('Average Win Rate')
axes[2].set_title('Evo Usage vs Win Rate')

plt.tight_layout()
plt.show()

## Prepare Data for ML Training

In [ ]:
def prepare_ml_training_data(simulation_df: pd.DataFrame) -> pd.DataFrame:
    """Prepare simulation data for machine learning model training"""
    
    # Create feature vectors for each deck configuration
    ml_features = []
    
    for deck_name in simulation_df['deck_name'].unique():
        deck_data = simulation_df[simulation_df['deck_name'] == deck_name]
        
        # Calculate deck composition features
        deck_cards = deck_presets[deck_name]
        theme_counts = {'AI': 0, 'Nano': 0, 'Quantum': 0, 'Base': 0}
        cost_distribution = {'0_cost': 0, '1_cost': 0, '2_cost': 0, '3_cost': 0, '4_plus_cost': 0}
        card_types = {'equipment': 0, 'item': 0, 'action': 0}
        
        for card in deck_cards:
            theme_counts[card.theme] += 1
            card_types[card.card_type.value] += 1
            
            # Cost analysis
            total_cost = sum(card.cost.values())
            if total_cost == 0:
                cost_distribution['0_cost'] += 1
            elif total_cost == 1:
                cost_distribution['1_cost'] += 1
            elif total_cost == 2:
                cost_distribution['2_cost'] += 1
            elif total_cost == 3:
                cost_distribution['3_cost'] += 1
            else:
                cost_distribution['4_plus_cost'] += 1
        
        # Simulation performance metrics
        performance_metrics = {
            'avg_win_rate': deck_data['win_rate'].mean(),
            'win_rate_std': deck_data['win_rate'].std(),
            'avg_turns': deck_data['turns_played'].mean(),
            'avg_evos': deck_data['evos_equipped'].mean(),
            'avg_teklo_energy': deck_data['teklo_energy_generated'].mean(),
            'avg_nanite_counters': deck_data['nanite_counters_generated'].mean(),
            'avg_quantum_charges': deck_data['quantum_charges_generated'].mean(),
            'avg_final_life': deck_data['teklovossen_final_life'].mean(),
            'consistency_score': 1.0 - deck_data['turns_played'].std() / deck_data['turns_played'].mean()
        }
        
        # Combine all features
        feature_vector = {
            'deck_name': deck_name,
            **theme_counts,
            **cost_distribution,
            **card_types,
            **performance_metrics
        }
        
        # Add expert balance scores if available
        expert_scores = []
        for card in deck_cards:
            if card.name in CARD_BALANCE_SCORES:
                scores = CARD_BALANCE_SCORES[card.name]
                expert_scores.append(scores['balance'])
        
        feature_vector.update({
            'avg_expert_balance_score': np.mean(expert_scores) if expert_scores else 5.0,
            'cards_with_expert_scores': len(expert_scores)
        })
        
        ml_features.append(feature_vector)
    
    ml_df = pd.DataFrame(ml_features)
    
    # Save ML training data
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    ml_df.to_csv(f"../data/processed/ml_training_data_{timestamp}.csv", index=False)
    
    return ml_df

# Prepare ML training data
ml_training_data = prepare_ml_training_data(simulation_results)
print("ML Training Data Prepared:")
print(ml_training_data)

# Feature correlation analysis
numeric_columns = ml_training_data.select_dtypes(include=[np.number]).columns
correlation_matrix = ml_training_data[numeric_columns].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix for ML Training')
plt.tight_layout()
plt.show()

print(f"\nML training data saved to data/processed/ml_training_data_*.csv")
print(f"Features available: {list(ml_training_data.columns)}")

## Summary and Next Steps

In [ ]:
# Generate summary report
summary = {
    'total_games_simulated': len(simulation_results),
    'deck_types_tested': simulation_results['deck_name'].nunique(),
    'average_game_length': simulation_results['turns_played'].mean(),
    'overall_teklovossen_win_rate': simulation_results['win_rate'].mean(),
    'best_performing_deck': simulation_results.groupby('deck_name')['win_rate'].mean().idxmax(),
    'most_consistent_deck': (
        simulation_results.groupby('deck_name')['turns_played'].std() / 
        simulation_results.groupby('deck_name')['turns_played'].mean()
    ).idxmin()
}

print("=== SIMULATION SUMMARY ===")
for key, value in summary.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

print("\n=== NEXT STEPS FOR ML ANALYSIS ===")
print("1. Use AWS SageMaker to train regression models predicting card balance")
print("2. Train classification models to predict deck archetype success")
print("3. Implement reinforcement learning for optimal play strategies")
print("4. Create card recommendation system based on simulation results")
print("5. Develop automated balance adjustment suggestions")

print("\n=== FILES GENERATED ===")
print("- Simulation data: data/simulations/batch_simulation_*.csv")
print("- ML training data: data/processed/ml_training_data_*.csv")
print("- Ready for AWS SageMaker integration")